# Serve Gnani Evon v3.3 for Bolo Khata (free Kaggle GPU)

Runs **Gnani Evon v3.3** (30B MoE, open weights) with Ollama on Kaggle's free **2× T4** GPUs and exposes an OpenAI-compatible URL through a Cloudflare tunnel.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Settings → **Internet on** (needs a phone-verified Kaggle account).

Then: Run All → copy the `https://….trycloudflare.com` URL printed at the end → in `C:Gnani.env` set `EVON_BASE_URL=https://….trycloudflare.com/v1` → restart Bolo Khata.

Keep this notebook running while you demo. A Kaggle GPU session lasts up to 12 hours (30 GPU-hours per week).

In [ ]:
# Model file. Q4_K_M (24.7 GB) splits across the two T4s. Use Q3_K_M (20 GB) if you run out of memory.
QUANT = "Q4_K_M"
MODEL = f"hf.co/luminenio/gnani-evon-v3.3-30B-A3B-GGUF:{QUANT}"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h / | tail -1

In [ ]:
# Install Ollama (bundles CUDA) and cloudflared
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!ollama --version; cloudflared --version

In [ ]:
import os, subprocess, time
env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="2",
           OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_SCHED_SPREAD="1")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s localhost:11434/api/version

In [ ]:
# Download Evon from Hugging Face (~25 GB, about 5–10 minutes)
!ollama pull {MODEL}
!ollama list

In [ ]:
# Warm up: loads the model onto the GPUs and checks JSON output
import json, urllib.request
req = urllib.request.Request("http://localhost:11434/v1/chat/completions", method="POST",
    headers={"Content-Type": "application/json"},
    data=json.dumps({"model": MODEL, "temperature": 0.1, "max_tokens": 200,
        "messages": [{"role": "system", "content": "Reply only with JSON: {\"customer\": string, \"amount\": number, \"type\": \"credit\"|\"payment\"}"},
                     {"role": "user", "content": "Ramesh bhaiya ko dhai sau ka atta udhaar likh do"}]}).encode())
t = time.time()
print(json.loads(urllib.request.urlopen(req, timeout=600).read())["choices"][0]["message"]["content"])
print(f"{time.time()-t:.1f}s")
!ollama ps

In [ ]:
# Public tunnel. Copy the trycloudflare URL into .env as EVON_BASE_URL=<url>/v1
import re
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+.trycloudflare.com", line)
    if m:
        url = m.group(0); break
print("
" + "=" * 70)
print(f"EVON_BASE_URL={url}/v1")
print(f"EVON_MODEL={MODEL}")
print("=" * 70)

In [ ]:
# Keep the session alive and show Evon's load status every 5 minutes
while True:
    !ollama ps
    time.sleep(300)